In [2]:
# Install necessary libraries
# pip install langchain pandas openai

import pandas as pd
from langchain.agents import initialize_agent, Tool
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain_experimental.tools import PythonREPLTool
from langchain.schema import AgentFinish
from langchain_groq import ChatGroq
# warnings.filterwarnings('ignore')
# load_dotenv()


In [3]:
class FileLoader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)

    def get_schema(self):
        """Return schema and a preview of the data."""
        return {
            "columns": self.data.columns.tolist(),
            "sample_rows": self.data.head(5).to_dict(orient='records'),
        }

    def execute_query(self, code: str):
        """Execute the Pandas code and return the result."""
        try:
            local_vars = {'df': self.data}
            exec(code, {}, local_vars)
            return local_vars.get("result", "No result variable found")
        except Exception as e:
            return f"Error in execution: {e}"


In [19]:
model = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="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"
    # other params...
)

def create_agent_a():
    # llm = ChatOpenAI(temperature=0)
    llm = model
    template = """
    You are a coding assistant. Given a schema and user query, generate Pandas code to answer the query.
    Schema: {schema}
    Query: {query}
    Output code should assign the result to a variable named `result`.
    """
    prompt = PromptTemplate(input_variables=["schema", "query"], template=template)
    # prompt = PromptTemplate()
    memory = ConversationBufferMemory()

    # Correctly initialize the PythonREPLTool with a description
    repl_tool = Tool(
        name="PythonREPL",
        func=PythonREPLTool(),
        description="A Python REPL tool for executing Python code."
    )
    
    return initialize_agent([repl_tool], llm, memory=memory, verbose=True, prompt=prompt)


In [20]:
def create_agent_b():
    def execute_code(code, file_loader):
        return file_loader.execute_query(code)

    return Tool(name="Code Executor", func=execute_code, description="Executes the generated code.")


In [21]:
def create_agent_c():
    # Create an LLM instance
    # llm = ChatOpenAI(temperature=0.3)
    llm = model
    
    # Define the prompt template
    template = """
    You are an assistant that generates insights based on data. Given the result of a query execution, return a natural language insight.
    Data: {data}
    Insight:
    """
    prompt = PromptTemplate(input_variables=["data"], template=template)

    # Initialize the tool
    def insight_tool(data):
        # The tool will return insights based on the data
        return f"Insight based on the provided data: {data}"

    # Return the agent with one simple tool
    return initialize_agent([Tool(name="InsightGenerator", func=insight_tool, description="Generates insights based on query results.")], llm, verbose=True, prompt=prompt)



In [22]:
file_loader = FileLoader(file_path= r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv" )
schema = file_loader.get_schema()
agent_a = create_agent_a()
agent_b = create_agent_b()
agent_c = create_agent_c()
code_response = agent_a.invoke({'input':{"schema":schema, "query": "Give me the number of columns?"}})



> Entering new AgentExecutor chain...
The question is asking for the number of columns in the given schema. I can find this by looking at the 'columns' key in the schema dictionary.

Action: PythonREPL
Action Input: 'len(schema["columns"])'
Observation: 
Thought:The observation is '9'. This is the number of columns in the schema.

Thought: I now know the final answer
Final Answer: The number of columns in the given schema is 9.


ValidationError: 2 validation errors for HumanMessage
content.str
  Input should be a valid string [type=string_type, input_value={'schema': {'columns': ['...the number of columns?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/string_type
content.list[union[str,dict[any,any]]]
  Input should be a valid list [type=list_type, input_value={'schema': {'columns': ['...the number of columns?'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/list_type

In [12]:
def main(file_path, user_query):
    # Load the file
    file_loader = FileLoader(file_path)
    schema = file_loader.get_schema()
    
    # Initialize agents
    agent_a = create_agent_a()
    agent_b = create_agent_b()
    agent_c = create_agent_c()
    
    # Generate code (Agent A)
    code_response = agent_a.invoke({"schema":schema, "query": user_query})
    
    # Execute code (Agent B)
    execution_result = agent_b.invoke({"code": code_response, "file_loader": file_loader})
    
    # Generate insights (Agent C)
    insight = agent_c.invoke({"data": execution_result})
    
    return insight



In [13]:
if __name__ == "__main__":
    file_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv"  # Path to your CSV/Excel file
    user_query = "Give me the mean of average price?"
    insight = main(file_path, user_query)
    print(insight)




> Entering new AgentExecutor chain...


ValueError: Missing some input keys: {'input'}

In [28]:
import pandas as pd
from langchain.prompts import PromptTemplate
from langchain.agents import initialize_agent
from langchain.llms import OpenAI
from langchain.tools import Tool

# Mock FileLoader for loading and schema extraction
class FileLoader:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_data(self):
        # Replace with actual logic for loading the file
        if self.file_path.endswith(".csv"):
            return pd.read_csv(self.file_path)
        elif self.file_path.endswith(".xlsx"):
            return pd.read_excel(self.file_path)
        else:
            raise ValueError("Unsupported file type")

    def get_schema(self, data):
        # Extract schema from DataFrame
        columns = data.columns.tolist()
        sample_rows = data.head(3).to_dict(orient="records")
        return {"columns": columns, "sample_rows": sample_rows}

# Agent A: Code generation based on schema and query
def create_agent_a():
    llm = model # Replace with your actual LLM initialization
    template = """
    You are an assistant for generating Python code to query tabular data.
    Schema: {schema}
    User Query: {query}
    Generate Python code using Pandas to answer the query:
    """
    prompt = PromptTemplate(input_variables=["schema", "query"], template=template)
    repl_tool = Tool(
        name="PythonREPL",
        func=PythonREPLTool(),
        description="A Python REPL tool for executing Python code."
    )
    tools = [repl_tool]  # Add tools if needed
    return initialize_agent(tools, llm, agent_type="zero-shot-react-description", verbose=True, prompt=prompt)

# Agent B: Executes the generated Python code in REPL
# def create_agent_b():
#     def execute_code(code, df):
#         # Execute code in sandboxed REPL
#         exec_globals = {"df": df, "result": None}
#         exec(code, exec_globals)
#         return exec_globals.get("result", "No result returned")

#     tool = Tool(
#         name="PythonExecutor",
#         func=execute_code,
#         description="Executes generated Python code using the provided DataFrame.",
#     )
#     return initialize_agent([tool], model, agent_type="zero-shot-react-description", verbose=True)

def create_agent_b():
    def execute_code(inputs):
        # Extract the `code` and `df` from the inputs dictionary
        code = inputs.get("code")
        df = inputs.get("df")
        
        if not code or df is None:
            return "Error: Missing 'code' or 'df' in inputs."

        # Execute code in sandboxed REPL
        exec_globals = {"df": df, "result": None}
        try:
            exec(code, exec_globals)
            return exec_globals.get("result", "No result returned")
        except Exception as e:
            return f"Execution error: {str(e)}"

    tool = Tool(
        name="PythonExecutor",
        func=execute_code,
        description="Executes generated Python code using the provided DataFrame.",
    )
    return initialize_agent([tool], model, agent_type="zero-shot-react-description", verbose=True)


# Agent C: Generates insights from execution results
def create_agent_c():
    llm = model  # Replace with your actual LLM initialization
    template = """
    You are an assistant that provides insights based on execution results.
    Execution Results: {data}
    Provide a detailed insight:
    """
    prompt = PromptTemplate(input_variables=["data"], template=template)
      # Add tools if needed
    def insight_tool(data):
        # The tool will return insights based on the data
        return f"Insight based on the provided data: {data}"
    tools = [insight_tool]
    # Return the agent with one simple tool
    return initialize_agent([Tool(name="InsightGenerator", func=insight_tool, description="Generates insights based on query results.")], llm, verbose=True, prompt=prompt)
    # return initialize_agent(tools, llm, agent_type="zero-shot-react-description", verbose=True, prompt=prompt)

# Main function to process the file and query
def main(file_path, user_query):
    # Load the file and schema
    file_loader = FileLoader(file_path)
    data = file_loader.load_data()
    schema = file_loader.get_schema(data)

    # Initialize agents
    agent_a = create_agent_a()
    agent_b = create_agent_b()
    agent_c = create_agent_c()

    # Step 1: Generate code (Agent A)
    code_response = agent_a.invoke({"input": {"schema": schema, "query": user_query}})
    print("Generated Code:\n", code_response)

    # Step 2: Execute the generated code (Agent B)
    execution_result = agent_b.invoke({"input": {"code": code_response, "df": data}})
    print("Execution Result:\n", execution_result)

    # Step 3: Generate insights (Agent C)
    insight = agent_c.invoke({"input": {"data": execution_result}})
    print("Insight:\n", insight)

    return insight

# Example usage
if __name__ == "__main__":
    file_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv"  
    user_query = "Give me the mean of average price?"
    insight = main(file_path, user_query)
    print("Final Insight:", insight)




> Entering new AgentExecutor chain...
To find the mean of the 'Average Price' column, I first need to extract this column from the given data. This data is provided in a dictionary format with a key 'sample_rows' that contains a list of dictionaries. Each dictionary represents a row of data and has a key 'Average Price' that I'm interested in.

Action: PythonREPL
Action Input: 
```python
average_prices = [row['Average Price'] for row in data['query']['sample_rows']]
mean_average_price = sum(average_prices) / len(average_prices)
mean_average_price
```
Observation: NameError("name 'data' is not defined")
Thought:It seems I made a mistake in the Action Input. I should have used the provided variable 'schema' instead of 'data'.

Action: PythonREPL
Action Input: 
```python
average_prices = [row['Average Price'] for row in schema['sample_rows']]
mean_average_price = sum(average_prices) / len(average_prices)
mean_average_price
```
Observation: NameError("name 'schema' is not defined")
Thoug

AttributeError: 'str' object has no attribute 'get'

- using sqlite db

In [126]:
import sqlite3
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
# Code Generator Agent (SQL Query Generator)
class CodeGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_sql_query(self, query, db_info, sample_records):
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are an assistant for generating SQL queries for an SQLite database.
                    The database schema and details are provided below:
                    Table name: dataTable
                    Schema: {db_info}
                    Sample records: {sample_records}
                    THE RESPONSE MUST BE STRICTLY THE SQL QUERY.
                    """,
                ),
                ("human", "User Query: {query}"),
            ]
        )

        # Use the LLM to generate SQL query
        chain = prompt | self.llm
        response = chain.invoke({"db_info": db_info, 
                                 "sample_records": sample_records,
                                 "query": query})
        return response.content.strip()  # Remove extra whitespace or newlines

# Code Executor Agent (SQL Query Executor)
class CodeExecutorAgent:
    def __init__(self, db_connection):
        self.db_connection = db_connection

    def execute_sql_query(self, sql_query):
        try:
            cursor = self.db_connection.cursor()
            cursor.execute(sql_query)
            result = cursor.fetchall()  # Fetch all results from the query execution
            return result
        except Exception as e:
            return f"Error executing SQL query: {str(e)}"

# Insight Generator Agent
class InsightGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_insight(self, user_query, sql_query, execution_result):
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are an assistant for converting the result into a natural language response to user's query.
                    The result is from executing an SQL query on an SQLite database, and you need to generate natural language response from it.
                    user query: {user_query}
                    sql query: {sql_query}
                    Execution Result: {execution_result}
                    Provide a natural language response or key insights based on the execution result.
                    
                    for e.g.
                    user query: "Give me the count of records in dataTable?"
                    sql query: SELECT COUNT(*) FROM dataTable
                    Execution Result: [(200,)]
                    Response: "The table contains 200 data points."
                    """,
                ),
            ]
        )

        # Generate insight from execution result
        chain = prompt | self.llm
        response = chain.invoke({"user_query": user_query,
                                 "sql_query": sql_query,
                                 "execution_result": execution_result})
        return response.content.strip()

# Convert CSV to SQLite Database
def csv_to_sqlite(csv_file_path, sqlite_db_path):
    # Load CSV into pandas DataFrame
    df = pd.read_csv(csv_file_path)
    # Create SQLite database and write the DataFrame to it
    conn = sqlite3.connect(sqlite_db_path)
    df.to_sql('dataTable', conn, if_exists='replace', index=False)
    sample_records = df.head(5).to_dict(orient="records")
    return conn, sample_records

# Main Logic
def main(user_query, csv_file_path, sqlite_db_path):
    # Step 1: Convert CSV to SQLite
    db_connection, sample_records = csv_to_sqlite(csv_file_path, sqlite_db_path)
    
    # Step 2: Get database schema (tables and column info)
    db_info = get_db_schema(db_connection)
    # print("database info: \n", db_info)
    # Initialize agents
    code_generator = CodeGeneratorAgent(llm=model)
    code_executor = CodeExecutorAgent(db_connection=db_connection)
    insight_generator = InsightGeneratorAgent(llm=model)

    # Step 3: Generate SQL Query (Agent A)
    generated_sql_query = code_generator.generate_sql_query(query=user_query, db_info=db_info, sample_records=sample_records)
    print("Generated SQL Query:\n", generated_sql_query)

    # Step 4: Execute the SQL query (Agent B)
    execution_result = code_executor.execute_sql_query(generated_sql_query)
    print("Execution Result:\n", execution_result)

    # Step 5: Generate insights from the execution result (Agent C)
    insight = insight_generator.generate_insight(user_query=user_query,sql_query=generated_sql_query, execution_result=execution_result)
    # print("Insight:\n", insight)

    return insight

# Get database schema
def get_db_schema(db_connection):
    cursor = db_connection.cursor()
    cursor.execute("PRAGMA table_info(dataTable);")
    schema_info = cursor.fetchall()
    db_info = "\n".join([f"Column: {col[1]}, Type: {col[2]}" for col in schema_info])
    
    return db_info




In [127]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="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"
    # other params...
)
if __name__ == "__main__":
    user_query = "Please tell me the mean of average price column?"
    csv_file_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv"  # Path to your CSV file
    sqlite_db_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.db"   # Path to the SQLite database

    # Call the main function
    insight = main(user_query, csv_file_path, sqlite_db_path)
    print("Final Insight:", insight)

Generated SQL Query:
 SELECT AVG(CAST(REPLACE(AveragePrice, '$', '') AS REAL)) FROM dataTable
Execution Result:
 [(200.7672,)]
Final Insight: The average price is approximately $200.77.


In [92]:
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
# Agent to generate Python code based on the user's query
class CodeGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_code(self, query, df_info, df_head=None):
        prompt = ChatPromptTemplate.from_messages(
                    [
                        (
                            "system",
                            """
                            You are an assistant for generating only Python code to query tabular data based on the user query. 
                            The code must be runnable on a DataFrame with no assumptions about its structure or variables.
                            The dataframe information is provided below:
                            DataFrame Info: {df_info}
                            The generated code should focus solely on the logic to answer the query without comments, explanations, or extra text. Do not include any starting text like ```python```, ```here is the code```, ```etc.```
                            """,
                        ),
                        ("human", "User Query: {query}"),
                    ]
                )

        # Use the LLM to generate code
        chain = prompt | self.llm
        response = chain.invoke({"df_info": df_info,
                                #  "df_head": df_head,
                                 "query": query})
        return response.content

# Agent to execute Python code on the DataFrame
class CodeExecutorAgent:
    
    def execute_code(self, code, df):
        exec_globals = {"df": df, "result": None}
        try:
            exec(code, exec_globals)
            return exec_globals.get("result", "No result returned")
        except Exception as e:
            return f"Error executing code: {e}"

# Agent to generate insights from the result
class InsightGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_insight(self, result):
        prompt = f"""
        You are an assistant that provides insights based on execution results.
        Execution Results: {result}
        Provide a detailed insight:
        """
        # Use the LLM to generate insights
        response = self.llm(prompt)
        return response.strip()

# Main processing function
def process_query(file_path, user_query, llm):
    # Load the CSV file into a DataFrame
    try:
        if file_path.endswith(".csv"):
            df = pd.read_csv(file_path)
        elif file_path.endswith(".xlsx"):
            df = pd.read_excel(file_path)
        else:
            raise ValueError("Unsupported file type")
    except Exception as e:
        return f"Error loading file: {e}"

    # Get DataFrame context
    df_info = df.info(buf=None)  # Schema of the DataFrame
    df_head = df.head().to_dict(orient="records")  # First few rows of the DataFrame

    # Initialize agents
    code_generator = CodeGeneratorAgent(llm)
    code_executor = CodeExecutorAgent()
    insight_generator = InsightGeneratorAgent(llm)

    # Step 1: Generate Python code
    generated_code = code_generator.generate_code(user_query, df_info, df_head)
    print("Generated Code:\n", generated_code)

    # Step 2: Execute the generated code
    execution_result = code_executor.execute_code(generated_code, df)
    print("Execution Result:\n", execution_result)

    # Step 3: Generate insights
    if isinstance(execution_result, str) and execution_result.startswith("Error"):
        return execution_result
    else:
        insight = insight_generator.generate_insight(execution_result)
        print("Generated Insight:\n", insight)
        return insight

# Example usage
# if __name__ == "__main__":
#     # Mock LLM function (replace this with your actual LLM call, such as OpenAI or LangChain model)
#     def mock_llm(prompt):
#         print("LLM Prompt:\n", prompt)
#         # Replace with real LLM logic
#         return "Mock Python code response"

#     file_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv"
#     user_query = "Give me the mean of average price?"
#     insight = process_query(file_path, user_query, model)
#     print("Final Insight:", insight)


In [94]:
file_path = r"C:\Users\rahul\Desktop\Offshore\PubSec-Info-Assistant-Offshore\app\backend\test_data\parts_inventory.csv"
user_query = "Give me the top 5 rows of data?"
try:
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith(".xlsx"):
        df = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file type")
except Exception as e:
    print(f"Error loading file: {e}")

# Get DataFrame context
import io
buffer = io.StringIO()
df.info(buf=buffer)  # Schema of the DataFrame
df_info = buffer.getvalue()
df_head = df.head().to_dict(orient="records") 

model = ChatGroq(
    model="llama-3.2-1b-preview",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="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"
    # other params...
)


    # Initialize agents
code_generator = CodeGeneratorAgent(llm=model)
code_executor = CodeExecutorAgent()
insight_generator = InsightGeneratorAgent(llm=model)

generated_code = code_generator.generate_code(user_query, df_info)
print("Generated Code:\n", generated_code)
generated_code = generated_code.replace("```", "").replace("python", "").strip()
# Step 2: Execute the generated code
execution_result = code_executor.execute_code(generated_code, df)
print("Execution Result:\n", execution_result)

Generated Code:
 ```python
top_5 = df.nlargest(5, 'Stock Level')
print(top_5)
```
                         Description  Stock Level  ROP  ROQ  Avg Turnover  \
66    Turret Mechanism M1126 Stryker          100    8    2      7.101864   
67                   Air Filter JLTV          100   10    9      8.177258   
199            Gun Barrel M2 Bradley          100    4    1      4.612049   
23   Periscope Assembly M2A3 Bradley           98    4    6      5.367218   
79         Water Pump M88A2 Hercules           98    5    9      1.412013   

     Avg Lead Time Average Price ABC Analysis XYZ Analysis         Location  
66               5      $298.29             C            Z  Tobruk Barracks  
67               7      $235.93             B            Y   Camp Pendleton  
199              9      $208.57             B            Z   Camp Pendleton  
23               5      $145.67             B            Y       Fort Bragg  
79               9      $258.18             B            X       

In [45]:
a = str(df.info())
print(a)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Description    200 non-null    object 
 1   Stock Level    200 non-null    int64  
 2   ROP            200 non-null    int64  
 3   ROQ            200 non-null    int64  
 4   Avg Turnover   200 non-null    float64
 5   Avg Lead Time  200 non-null    int64  
 6   Average Price  200 non-null    object 
 7   ABC Analysis   200 non-null    object 
 8   XYZ Analysis   200 non-null    object 
 9   Location       200 non-null    object 
dtypes: float64(1), int64(4), object(5)
memory usage: 15.8+ KB
None
